# EvoBrain: Multi-Horizon Future Prediction & Seizure Classification on GPU

This notebook trains **EvoBrain** and its baselines with a **Dual-Head Multi-Task Objective**:
1. **Head 1 (Time Proximity)**: Predicts **when** a seizure occurs across 5 horizons:
   - `now` : Seizure active right now
   - `5s`  : Onset within next 5 seconds (Imminent)
   - `10s` : Onset within next 10 seconds (Warning)
   - `20s` : Onset within next 20 seconds (Alert)
   - `40s` : Onset within next 40 seconds (Advisory)
   - **Delta t > 40s**: Strictly set to **0** (`[0, 0, 0, 0, 0]` = Safe Background)
2. **Head 2 (Seizure Subtype)**: Predicts **what kind** of seizure it is / will be:
   - `0: bckg` (Normal baseline brainwaves)
   - `1: fnsz` (Focal Non-Specific Seizure)
   - `2: gnsz` (Generalized Non-Specific Seizure)
   - `3: cpsz` (Complex Partial Seizure)
   - `4: other` (Tonic-Clonic, Absence, Myoclonic, etc.)

### Models Supported
- `lstm_baseline`: Fast sequential temporal baseline.
- `gru_gcn_baseline`: Spatial-temporal GCN with GRU (**FPGA target**).
- `evobrain_gru`: EvoBrain with GRU backbone.
- `og_evobrain`: Original NeurIPS 2025 Spotlight (Two-Stream Mamba + Laplacian PE + SSGConv).

In [ ]:
# @title 1. GPU Check & Environment Setup
import torch
import sys
import os

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"Device Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")

# Install dependencies
!pip install -q torch-geometric datasets huggingface_hub pyedflib scikit-learn more-itertools h5py


PyTorch Version : 2.11.0+cu130
CUDA Available  : True
Device Name     : NVIDIA L4
Device Memory   : 23.66 GB
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# Source - https://stackoverflow.com/a/1006337
# Posted by Nadia Alramli, modified by community. See post 'Timeline' for change history
# Retrieved 2026-10-28, License - CC BY-SA 3.0

import multiprocessing

multiprocessing.cpu_count()


12

In [ ]:
# @title 2. Mount Google Drive & Clone EvoBrain Codebase
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/EvoBrain'):
    !git clone https://github.com/Kotoge/EvoBrain.git /content/EvoBrain

sys.path.append('/content/EvoBrain')
os.chdir('/content/EvoBrain')
print("Working directory set to /content/EvoBrain")

Mounted at /content/drive
Cloning into '/content/EvoBrain'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 66 (delta 1), reused 0 (delta 0), pack-reused 53 (from 1)
Receiving objects: 100% (66/66), 2.02 MiB | 1.46 MiB/s, done.
Resolving deltas: 100% (26/26), done.
Working directory set to /content/EvoBrain


In [ ]:
# @title 3. Dataset: Soft Time Horizons [now, 5s, 10s, 20s, 40s] + Seizure Class
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from scipy.fftpack import fft
import numpy as np
import json

HF_REPO_ID = "Shio-Koube/tusz-evobrain-19ch-200hz"  # @param {type:"string"}
BATCH_SIZE = 128                                       # @param {type:"integer"}
NUM_WORKERS = 4                                        # @param {type:"integer"}
TOP_K = 3                                              # Sparsity for dynamic graph

# Seizure Subtype Map
CLASS_MAP = {"bckg": 0, "fnsz": 1, "gnsz": 2, "cpsz": 3}

try:
    hf_train = load_dataset(HF_REPO_ID, split="train")
    hf_dev = load_dataset(HF_REPO_ID, split="validation")
    print(f"Loaded HF train: {len(hf_train)} recordings | dev: {len(hf_dev)} recordings")
except Exception as e:
    print(f"Could not load from Hugging Face: {e}")

def compute_step_fft(signal_1s, n_fft=200):
    fourier = fft(signal_1s, n=n_fft, axis=-1)
    amp = np.abs(fourier[:, :int(n_fft // 2)])  # 100 positive frequency bins
    amp[amp == 0.0] = 1e-8
    return np.log(amp)

def compute_step_adjacency(step_raw, top_k=TOP_K):
    norms = np.linalg.norm(step_raw, axis=-1, keepdims=True)
    norms[norms == 0.0] = 1e-8
    normed = step_raw / norms
    corr = np.abs(normed @ normed.T)
    np.fill_diagonal(corr, 1.0)

    corr_no_self = corr.copy()
    np.fill_diagonal(corr_no_self, 0.0)
    topk_idx = (-corr_no_self).argsort(axis=-1)[:, :top_k]
    mask = np.eye(corr.shape[0], dtype=bool)
    for i in range(corr.shape[0]):
        mask[i, topk_idx[i]] = True
    return (mask * corr).astype(np.float32)

def compute_time_and_class_targets(w_start, w_end, events_bi, events_multi):
    """
    Generates 5-horizon soft time vector: [now, 5s, 10s, 20s, 40s].
    When delta_t > 40s, all time targets are strictly 0.0!
    """
    time_target = np.zeros(5, dtype=np.float32)  # [now, 5s, 10s, 20s, 40s]
    class_target = 0                             # 0 = bckg

    for ev in events_bi:
        s_on, s_off = ev["start"], ev["stop"]

        # 1. Active seizure occurring right now
        if max(w_start, s_on) < min(w_end, s_off):
            time_target[0] = 1.0  # now
            time_target[1:] = 1.0
            break

        # 2. Upcoming seizure in the future
        elif s_on > w_end:
            delta_t = s_on - w_end
            if delta_t <= 40.0:
                # Exponential soft labels for smooth transition
                time_target[4] = float(np.exp(-delta_t / 40.0))  # 40s
                if delta_t <= 20.0:
                    time_target[3] = float(np.exp(-delta_t / 20.0))  # 20s
                if delta_t <= 10.0:
                    time_target[2] = float(np.exp(-delta_t / 10.0))  # 10s
                if delta_t <= 5.0:
                    time_target[1] = float(np.exp(-delta_t / 5.0))   # 5s
                break
            # When delta_t > 40s, time_target remains [0, 0, 0, 0, 0]!

    # Determine seizure subtype if active or within warning window
    if time_target.max() > 0.25:
        for ev in events_multi:
            subtype = ev.get("label", "other")
            class_target = CLASS_MAP.get(subtype, 4)  # 4 = other
            break

    return time_target, class_target

class EvoBrainMultiTaskDataset(Dataset):
    def __init__(self, hf_ds, clip_len_sec=12, stride_sec=4, is_train=True):
        self.hf_ds = hf_ds
        self.clip_len_sec = clip_len_sec
        self.stride_sec = stride_sec
        self.samples = []

        ds_meta = hf_ds.remove_columns(["eeg_fp16_bytes"])
        for row_idx in range(len(ds_meta)):
            row = ds_meta[row_idx]
            sfreq = row["sfreq"]
            n_samples = row["n_samples"]
            win_samples = int(clip_len_sec * sfreq)
            stride_samples = int(stride_sec * sfreq)
            if n_samples < win_samples:
                continue

            events_bi = json.loads(row["events_bi_json"])
            events_multi = json.loads(row["events_multi_json"])

            for start_s in range(0, n_samples - win_samples + 1, stride_samples):
                end_s = start_s + win_samples
                w_start = start_s / sfreq
                w_end = end_s / sfreq

                t_target, c_target = compute_time_and_class_targets(w_start, w_end, events_bi, events_multi)
                self.samples.append((row_idx, start_s, end_s, t_target, c_target))

        # Class balancing for training set (balance positive warning/seizure vs pure background)
        if is_train:
            pos = [s for s in self.samples if s[3].max() > 0.0]
            neg = [s for s in self.samples if s[3].max() == 0.0]
            np.random.shuffle(neg)
            self.samples = pos + neg[:len(pos)]
            np.random.shuffle(self.samples)
            print(f"Train Balanced: {len(pos)} Warning/Seizure clips + {len(pos)} Background clips = {len(self.samples)} Total")
        else:
            print(f"Dev/Val Set  : {len(self.samples)} Total clips")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row_idx, start_s, end_s, t_target, c_target = self.samples[idx]
        row = self.hf_ds[row_idx]
        sfreq = row["sfreq"]
        eeg = np.frombuffer(row["eeg_fp16_bytes"], dtype=np.float16).reshape(19, row["n_samples"])
        clip = eeg[:, start_s:end_s].astype(np.float32)

        fft_steps = []
        adj_steps = []
        step_samples = int(sfreq)
        for t in range(self.clip_len_sec):
            step_raw = clip[:, t*step_samples : (t+1)*step_samples]
            fft_steps.append(compute_step_fft(step_raw, n_fft=step_samples))
            adj_steps.append(compute_step_adjacency(step_raw, top_k=TOP_K))

        x_fft = np.stack(fft_steps, axis=0)   # (12, 19, 100)
        adj_seq = np.stack(adj_steps, axis=0) # (12, 19, 19)

        return {
            "x": torch.from_numpy(x_fft).float(),
            "adj": torch.from_numpy(adj_seq).float(),
            "time_target": torch.from_numpy(t_target).float(),  # (5,) [now, 5s, 10s, 20s, 40s]
            "class_target": torch.tensor(c_target).long(),       # (scalar) 0..4
            "seq_len": torch.tensor(self.clip_len_sec).long()
        }

train_ds = EvoBrainMultiTaskDataset(hf_train, clip_len_sec=12, stride_sec=3, is_train=True)
dev_ds = EvoBrainMultiTaskDataset(hf_dev, clip_len_sec=12, stride_sec=6, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print("Multi-Horizon DataLoaders ready!")

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

data/train/train-shard-0000-of-0021.parq(…): reconstructing file:   0%|          |  0.00B /  278MB            

data/train/train-shard-0000-of-0021.parq(…): downloading bytes:           |  0.00B            

data/train/train-shard-0001-of-0021.parq(…): reconstructing file:   0%|          |  0.00B /  209MB            

data/train/train-shard-0001-of-0021.parq(…): downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/1179 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/427 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/229 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loaded HF train: 1179 recordings | dev: 427 recordings
Train Balanced: 59089 Warning/Seizure clips + 59089 Background clips = 118178 Total
Dev/Val Set  : 60594 Total clips
Multi-Horizon DataLoaders ready!


In [ ]:
# @title 4. Multi-Task Model (Dual Heads: Time Proximity + Class Subtype)
import torch.nn as nn
from types import SimpleNamespace

MODEL_CHOICE = "gru_gcn_baseline"  # @param ["lstm_baseline", "gru_gcn_baseline", "evobrain_gru", "og_evobrain"]
MODEL_SCALE = "small_108K"         # @param ["small_108K", "medium_1.5M", "large_5.5M", "xlarge_18M"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SCALE_PRESETS = {
    "small_108K":  dict(embed_dim=100, rnn_units=64,  num_rnn_layers=1, agg="max",    mlp_dim=0,    skip=False),
    "medium_1.5M": dict(embed_dim=256, rnn_units=256, num_rnn_layers=2, agg="concat", mlp_dim=256,  skip=True),
    "large_5.5M":  dict(embed_dim=384, rnn_units=384, num_rnn_layers=2, agg="concat", mlp_dim=512,  skip=True),
    "xlarge_18M":  dict(embed_dim=512, rnn_units=512, num_rnn_layers=3, agg="concat", mlp_dim=1024, skip=True),
}
cfg = SCALE_PRESETS[MODEL_SCALE]

args = SimpleNamespace(
    num_nodes=19,
    input_dim=100,
    embed_dim=cfg["embed_dim"],
    rnn_units=cfg["rnn_units"],
    num_rnn_layers=cfg["num_rnn_layers"],
    num_classes=64,
    dropout=0.2,
    max_seq_len=12,
    agg=cfg["agg"],
    mlp_dim=cfg["mlp_dim"],
    skip=cfg["skip"],
)

def make_head(in_dim, mlp_dim, out_dim, dropout=0.2):
    """FPGA-friendly MLP head using only BatchNorm1d (foldable into constants) + ReLU (sign-bit gate)."""
    if mlp_dim <= 0:
        return nn.Linear(in_dim, out_dim)
    return nn.Sequential(
        nn.Linear(in_dim, mlp_dim, bias=False),
        nn.BatchNorm1d(mlp_dim),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(mlp_dim, mlp_dim // 2, bias=False),
        nn.BatchNorm1d(mlp_dim // 2),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(mlp_dim // 2, out_dim),
    )

class MultiTaskSeizureModel(nn.Module):
    def __init__(self, backbone_model, hidden_dim=None, mlp_dim=512, num_classes=5):
        super().__init__()
        self.backbone = backbone_model
        if hidden_dim is None:
            hidden_dim = backbone_model.fc.in_features
        self.hidden_dim = hidden_dim

        # Static-foldable BatchNorm1d + ReLU on backbone output
        self.backbone_bn = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.2)

        # Head 1: Time Proximity across 5 horizons [now, 5s, 10s, 20s, 40s]
        self.time_head = make_head(hidden_dim, mlp_dim, 5)
        # Head 2: Seizure Subtype [bckg, fnsz, gnsz, cpsz, other]
        self.class_head = make_head(hidden_dim, mlp_dim, num_classes)

    def forward(self, x, seq_lens, adj=None):
        if adj is not None:
            _, hidden = self.backbone(x, seq_lens, adj)
        else:
            _, hidden = self.backbone(x, seq_lens)

        if hidden.dim() == 3:
            hidden = torch.max(hidden, dim=1)[0]  # (B, hidden_dim)

        feat = self.dropout(self.relu(self.backbone_bn(hidden)))
        time_logits = self.time_head(feat)    # (B, 5)
        class_logits = self.class_head(feat)  # (B, 5)
        return time_logits, class_logits

# Instantiate chosen backbone
if MODEL_CHOICE == "lstm_baseline":
    from model.lstm import LSTMModel
    base_model = LSTMModel(args, num_classes=64, device=device)
elif MODEL_CHOICE == "gru_gcn_baseline":
    from model.gru_gcn import GRU_GCN_classification, GRU_GCN
    base_model = GRU_GCN_classification(args=args, num_classes=64, device=device)
    # Upgrade internal GRU_GCN width, depth, and skip connections
    base_model.gru_gcn = GRU_GCN(
        feat_input_size_edge=args.input_dim,
        feat_input_size_node=args.input_dim,
        feat_target_size=args.rnn_units,
        embed_inside_size=args.embed_dim,
        convolve="gcn",
        reduce_edge="gru",
        reduce_node="gru",
        skip=args.skip,
        activate="tanh",
        concat=True,
        neo_gnn=True,
    )
    # Replace internal GNN Tanh/Softplus with hardware-free ReLU
    base_model.gru_gcn.activate = nn.ReLU()
    base_model.gru_gcn.gnnx2.activate = nn.ReLU()
    base_model.gru_gcn.gnnx2.edge_activate = nn.ReLU()
    if args.num_rnn_layers > 1:
        base_model.gru_gcn.snn_node = nn.GRU(args.input_dim, args.embed_dim, num_layers=args.num_rnn_layers, dropout=args.dropout)
        base_model.gru_gcn.snn_edge = nn.GRU(1, args.embed_dim, num_layers=args.num_rnn_layers, dropout=args.dropout)
    fc_in = base_model.gru_gcn.feat_target_size * (args.num_nodes if args.agg == "concat" else 1)
    base_model.fc = nn.Linear(fc_in, 64)
elif MODEL_CHOICE == "evobrain_gru":
    from model.EvoBrain import EvoBrain_classification, sequentialize
    base_model = EvoBrain_classification(args=args, num_classes=64, device=device, gnn="ssg", num_eigenvectors=8)
    base_model.evobrain.reduce_node = "gru"
    base_model.evobrain.reduce_edge = "gru"
    base_model.evobrain.activate = nn.ReLU()
    base_model.evobrain.edge_activate = nn.ReLU()
    base_model.evobrain.gnnx2.activate = nn.ReLU()
    base_model.evobrain.snn_node = nn.GRU(args.input_dim, args.input_dim, num_layers=args.num_rnn_layers, dropout=args.dropout if args.num_rnn_layers > 1 else 0.0).to(device)
    base_model.evobrain.snn_edge = nn.GRU(1, args.input_dim, num_layers=args.num_rnn_layers, dropout=args.dropout if args.num_rnn_layers > 1 else 0.0).to(device)
    fc_in = (base_model.evobrain.feat_target_size + 8) * (args.num_nodes if args.agg == "concat" else 1)
    base_model.fc = nn.Linear(fc_in, 64)
elif MODEL_CHOICE == "og_evobrain":
    from model.EvoBrain import EvoBrain_classification
    base_model = EvoBrain_classification(args=args, num_classes=64, device=device, gnn="ssg", num_eigenvectors=8)
    base_model.evobrain.activate = nn.ReLU()
    base_model.evobrain.edge_activate = nn.ReLU()
    base_model.evobrain.gnnx2.activate = nn.ReLU()
    fc_in = (base_model.evobrain.feat_target_size + 8) * (args.num_nodes if args.agg == "concat" else 1)
    base_model.fc = nn.Linear(fc_in, 64)

backbone_dim = base_model.fc.in_features
model = MultiTaskSeizureModel(base_model, hidden_dim=backbone_dim, mlp_dim=args.mlp_dim, num_classes=5).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Initialized: [{MODEL_CHOICE}] | Scale: [{MODEL_SCALE}] | Feature Dim: {backbone_dim}")
print(f"Total Trainable Parameters: {total_params:,} (~{total_params * 4 / (1024**2):.2f} MB in FP32)")

feat_input_size_edge: 1
feat_input_size_node: 100
embed_inside_size: 100
feat_target_size: 64
feat_input_size_edge: 1
feat_input_size_node: 100
embed_inside_size: 100
feat_target_size: 64
Model Initialized: [gru_gcn_baseline] | Scale: [small_108K] | Feature Dim: 164
Total Trainable Parameters: 127,167 (~0.49 MB in FP32)


In [ ]:
# @title 5. Multi-Task Training Loop (Time BCE + Class CrossEntropy)
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm.auto import tqdm

NUM_EPOCHS = 100
LEARNING_RATE = 1e-3
TIME_LOSS_WEIGHT = 1.0
CLASS_LOSS_WEIGHT = 0.5
CHECKPOINT_DIR = "/content/drive/MyDrive/evobrain_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

criterion_time = nn.BCEWithLogitsLoss()
criterion_class = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

best_auroc = 0.0
train_losses, val_aurocs_now, val_aurocs_10s = [], [], []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Train]")
    for batch in pbar:
        x = batch["x"].to(device)             # (B, 12, 19, 100)
        adj = batch["adj"].to(device)         # (B, 12, 19, 19)
        t_target = batch["time_target"].to(device)  # (B, 5)
        c_target = batch["class_target"].to(device) # (B,)
        seq_lens = batch["seq_len"].to(device)

        optimizer.zero_grad()
        if MODEL_CHOICE == "lstm_baseline":
            t_logits, c_logits = model(x, seq_lens)
        else:
            t_logits, c_logits = model(x, seq_lens, adj)

        loss_time = criterion_time(t_logits, t_target)
        loss_class = criterion_class(c_logits, c_target)
        loss = (TIME_LOSS_WEIGHT * loss_time) + (CLASS_LOSS_WEIGHT * loss_class)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "t_loss": f"{loss_time.item():.4f}"})

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    # --- Validation Phase ---
    model.eval()
    all_t_preds, all_t_targets = [], []
    all_c_preds, all_c_targets = [], []
    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Val]"):
            x = batch["x"].to(device)
            adj = batch["adj"].to(device)
            t_target = batch["time_target"].to(device)
            c_target = batch["class_target"].to(device)
            seq_lens = batch["seq_len"].to(device)

            if MODEL_CHOICE == "lstm_baseline":
                t_logits, c_logits = model(x, seq_lens)
            else:
                t_logits, c_logits = model(x, seq_lens, adj)

            t_probs = torch.sigmoid(t_logits).cpu().numpy()
            c_probs = torch.softmax(c_logits, dim=-1).cpu().numpy()

            all_t_preds.extend(t_probs)
            all_t_targets.extend(t_target.cpu().numpy())
            all_c_preds.extend(c_probs.argmax(axis=-1))
            all_c_targets.extend(c_target.cpu().numpy())

    all_t_preds = np.array(all_t_preds)      # (N, 5)
    all_t_targets = np.array(all_t_targets)  # (N, 5)

    # Compute AUROC for 'now' and '10s warning'
    auroc_now = roc_auc_score((all_t_targets[:, 0] > 0.5).astype(int), all_t_preds[:, 0])
    auroc_10s = roc_auc_score((all_t_targets[:, 2] > 0.2).astype(int), all_t_preds[:, 2])
    class_acc = accuracy_score(all_c_targets, all_c_preds)

    val_aurocs_now.append(auroc_now)
    val_aurocs_10s.append(auroc_10s)
    scheduler.step(auroc_now)

    print(f"Epoch {epoch:02d}: Loss={avg_loss:.4f} | AUROC Now={auroc_now:.4f} | AUROC 10s Future={auroc_10s:.4f} | Class Acc={class_acc:.4f}")

    if auroc_now > best_auroc:
        best_auroc = auroc_now
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"{MODEL_CHOICE}_multitask_best.pt")
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "best_auroc": best_auroc}, ckpt_path)
        print(f"  ==> Saved Checkpoint to {ckpt_path}")

Epoch 1/100 [Train]:   0%|          | 0/924 [00:00<?, ?it/s]

Epoch 1/100 [Val]:   0%|          | 0/474 [00:00<?, ?it/s]

Epoch 01: Loss=1.0497 | AUROC Now=0.8184 | AUROC 10s Future=0.8091 | Class Acc=0.4752
  ==> Saved Checkpoint to /content/drive/MyDrive/evobrain_checkpoints/gru_gcn_baseline_multitask_best.pt


Epoch 2/100 [Train]:   0%|          | 0/924 [00:00<?, ?it/s]

Epoch 2/100 [Val]:   0%|          | 0/474 [00:00<?, ?it/s]

...

Epoch 11: Loss=0.8219 | AUROC Now=0.8568 | AUROC 10s Future=0.8412 | Class Acc=0.8024


Epoch 12/100 [Train]:   0%|          | 0/924 [00:00<?, ?it/s]

Epoch 12/100 [Val]:   0%|          | 0/474 [00:00<?, ?it/s]

Epoch 12: Loss=0.8143 | AUROC Now=0.8660 | AUROC 10s Future=0.8461 | Class Acc=0.7947
  ==> Saved Checkpoint to /content/drive/MyDrive/evobrain_checkpoints/gru_gcn_baseline_multitask_best.pt


Epoch 13/100 [Train]:   0%|          | 0/924 [00:00<?, ?it/s]

In [ ]:
# @title 6. Plot Multi-Horizon Prediction Trajectories
import matplotlib.pyplot as plt

horizons = ["Now (Active)", "< 5s Onset", "< 10s Onset", "< 20s Onset", "< 40s Onset"]
auroc_list = []
for h_idx in range(5):
    y_bin = (all_t_targets[:, h_idx] > 0.2).astype(int)
    if len(np.unique(y_bin)) > 1:
        score = roc_auc_score(y_bin, all_t_preds[:, h_idx])
    else:
        score = 0.5
    auroc_list.append(score)

plt.figure(figsize=(9, 4.5))
plt.bar(horizons, auroc_list, color=["#dc2626", "#ea580c", "#eab308", "#16a34a", "#2563eb"])
plt.ylim(0.5, 1.0)
plt.ylabel("AUROC Score")
plt.title(f"Multi-Horizon Early Warning Accuracy ({MODEL_CHOICE})")
for i, v in enumerate(auroc_list):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# @title 7. (Bonus) Export Dual-Head Weights for FPGA Etching
# Exports the trained dual-head weights (Time Proximity + Seizure Class) to C++ header:
best_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, f"{MODEL_CHOICE}_multitask_best.pt"), map_location="cpu")
state_dict = best_ckpt["model_state_dict"]

header_path = os.path.join(CHECKPOINT_DIR, f"{MODEL_CHOICE}_etched_multitask_weights.h")
with open(header_path, "w") as f:
    f.write("// Auto-generated Etched Weights for FPGA Acceleration\n")
    f.write("// Architecture: " + MODEL_CHOICE + " (Multi-Horizon Time + Class)\n")
    f.write("#ifndef ETCHED_WEIGHTS_H\n#define ETCHED_WEIGHTS_H\n\n")

    for name, tensor in state_dict.items():
        sanitized_name = name.replace(".", "_")
        arr = tensor.numpy().flatten()
        f.write(f"// Parameter: {name} (Shape: {list(tensor.shape)})\n")
        f.write(f"const float {sanitized_name}[{len(arr)}] = {{\n")
        chunks = [", ".join(f"{v:.6f}f" for v in arr[i:i+8]) for i in range(0, len(arr), 8)]
        f.write("    " + ",\n    ".join(chunks) + "\n")
        f.write("};\n\n")

    f.write("#endif // ETCHED_WEIGHTS_H\n")

print(f"Successfully exported hardwired multi-task weights header to: {header_path}")